# Predicting F1 Pit Stops — Optimal Pipeline (v2)
**Playground Series S6E5** | Target: `PitNextLap` | Metric: ROC-AUC

**Key insights driving the design**:
- Train and test share all 104 races (row-level random split → StratifiedKFold matches LB).
- `TyreLife` is present (raw); `Normalized_TyreLife` was removed.
- All test inputs are visible at inference, so neighbor features computed over `train+test` are *legal*.
- When `lap_gap_next == 1`, `PitStop_next` perfectly determines `PitNextLap` — encoded as `oracle_pit_next`.

**v1 → v2 changes**: lag-2/lead-2 features, explicit oracle column, cumulative race-level pit counts, rolling pace slope, weighted blend by OOF AUC².

In [2]:
import os, gc, warnings, time
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata
import lightgbm as lgb
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)
SEED, N_FOLDS, DATA = 42, 5, 'data'

## 1. Load

In [3]:
train = pd.read_csv(f'{DATA}/train.csv')
test  = pd.read_csv(f'{DATA}/test.csv')
train.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
test.rename(columns={'LapTime (s)': 'LapTime'}, inplace=True)
TARGET, ID = 'PitNextLap', 'id'
print(train.shape, test.shape)
train.head(3)

(439140, 16) (188165, 15)


,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime,LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0


## 2. Feature Engineering

In [4]:
train['is_test'] = 0
test['is_test']  = 1
test[TARGET] = np.nan
full = pd.concat([train, test], ignore_index=True)
full = full.sort_values(['Race','Year','Driver','LapNumber']).reset_index(drop=True)
G  = ['Race','Year','Driver']
GS = ['Race','Year','Driver','Stint']
RACE = ['Race','Year']
print('full:', full.shape)

full: (627305, 17)


In [5]:
# 2a. Lag/lead within (Race, Year, Driver) — sorted by LapNumber
g = full.groupby(G, sort=False)

for col in ['TyreLife','LapTime','Position','Stint','Compound',
            'PitStop','Cumulative_Degradation','LapTime_Delta',
            'Position_Change','LapNumber']:
    full[f'{col}_prev'] = g[col].shift(1)
    full[f'{col}_next'] = g[col].shift(-1)

for col in ['TyreLife','LapTime','Stint','Compound','PitStop','LapNumber']:
    full[f'{col}_prev2'] = g[col].shift(2)
    full[f'{col}_next2'] = g[col].shift(-2)

full['lap_gap_prev']  = full['LapNumber'] - full['LapNumber_prev']
full['lap_gap_next']  = full['LapNumber_next']  - full['LapNumber']
full['lap_gap_next2'] = full['LapNumber_next2'] - full['LapNumber']
full['lap_gap_prev2'] = full['LapNumber'] - full['LapNumber_prev2']

# Tyre-life dynamics
full['tyre_life_delta_next']  = full['TyreLife_next']  - full['TyreLife']
full['tyre_life_delta_next2'] = full['TyreLife_next2'] - full['TyreLife']
full['tyre_life_delta_prev']  = full['TyreLife'] - full['TyreLife_prev']

full['stint_change_next']    = (full['Stint_next']  > full['Stint']).astype('float')
full['stint_change_next2']   = (full['Stint_next2'] > full['Stint']).astype('float')
full['stint_change_prev']    = (full['Stint'] > full['Stint_prev']).astype('float')
full['compound_change_next'] = (full['Compound_next'] != full['Compound']).astype('float')
full['compound_change_next2']= (full['Compound_next2']!= full['Compound']).astype('float')
full['just_pitted']          = (full['stint_change_prev'] == 1).astype('float')

# *** Direct target oracle: when next observed lap is exactly N+1, PitStop_next IS PitNextLap ***
full['oracle_pit_next'] = np.where(full['lap_gap_next'] == 1,
                                    full['PitStop_next'], np.nan)
full['oracle_gap2_pit'] = np.where(
    (full['lap_gap_next'] == 2) & (full['Stint_next'] > full['Stint']),
    1.0, np.nan
)

# Pace dynamics
full['laptime_diff_prev']  = full['LapTime'] - full['LapTime_prev']
full['laptime_diff_next']  = full['LapTime_next'] - full['LapTime']
full['laptime_diff_prev2'] = full['LapTime'] - full['LapTime_prev2']
full['pos_diff_prev']      = full['Position'] - full['Position_prev']
full['pos_diff_next']      = full['Position_next'] - full['Position']

# Rolling pace slope (excludes current lap — no leak)
full['laptime_roll3_mean'] = g['LapTime'].transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
full['laptime_vs_recent']  = full['LapTime'] - full['laptime_roll3_mean']
print('after lag/lead:', full.shape)

after lag/lead: (627305, 71)


In [6]:
# 2b. Stint-level aggregates (inputs only, no target leak)
gs = full.groupby(GS, sort=False)
full['stint_lap_min']      = gs['LapNumber'].transform('min')
full['stint_lap_max']      = gs['LapNumber'].transform('max')
full['stint_lap_count']    = gs['LapNumber'].transform('count')
full['stint_tyre_min']     = gs['TyreLife'].transform('min')
full['stint_tyre_max']     = gs['TyreLife'].transform('max')
full['stint_progress']     = (full['LapNumber'] - full['stint_lap_min']) / (full['stint_lap_max'] - full['stint_lap_min'] + 1)
full['laps_left_in_stint'] = full['stint_lap_max'] - full['LapNumber']
full['tyre_left_in_stint'] = full['stint_tyre_max'] - full['TyreLife']
full['stint_laptime_mean'] = gs['LapTime'].transform('mean')
full['laptime_vs_stint']   = full['LapTime'] - full['stint_laptime_mean']
full['stint_deg_max']      = gs['Cumulative_Degradation'].transform('max')
full['deg_pct_of_max']     = full['Cumulative_Degradation'] / (full['stint_deg_max'] + 1e-6)
# *** key: am I at the last observed lap of my stint? Strong pit signal ***
full['is_last_obs_of_stint'] = (full['LapNumber'] == full['stint_lap_max']).astype('float')
print('after stint agg:', full.shape)

after stint agg: (627305, 84)


In [7]:
# 2c. Compound × typical max TyreLife (rebuild the missing Normalized_TyreLife)
tr_mask = full['is_test'] == 0
cm = full[tr_mask].groupby('Compound')['TyreLife'].quantile(0.95).rename('compound_typical_max')
full = full.merge(cm, on='Compound', how='left')
full['tyre_life_normalised'] = full['TyreLife'] / (full['compound_typical_max'] + 1e-6)
full['tyre_life_over_max']   = (full['TyreLife'] > full['compound_typical_max']).astype('float')

csl = full[tr_mask].groupby('Compound')['stint_lap_count'].mean().rename('compound_avg_stint_len')
full = full.merge(csl, on='Compound', how='left')
full['stint_len_vs_compound_avg'] = full['stint_lap_count'] - full['compound_avg_stint_len']
print('after compound norm:', full.shape)

after compound norm: (627305, 89)


In [8]:
# 2d. Race-level context (field-wide signals at this lap)
rl = full.groupby(['Race','Year','LapNumber'])
full['n_drivers_this_lap']      = rl['Driver'].transform('count')
full['mean_position_this_lap']  = rl['Position'].transform('mean')
full['mean_laptime_this_lap']   = rl['LapTime'].transform('mean')
full['laptime_vs_field']        = full['LapTime'] - full['mean_laptime_this_lap']
full['field_pitrate_this_lap']  = rl['PitStop'].transform('mean')

rlc = full.groupby(['Race','Year','LapNumber','Compound'])
full['n_same_compound_this_lap'] = rlc['Driver'].transform('count')
full['frac_same_compound'] = full['n_same_compound_this_lap'] / full['n_drivers_this_lap']

# Field pit rate in upcoming 2 laps for this race (pressure to pit / safety car proxy)
ru = full.groupby(['Race','Year','LapNumber'])['PitStop'].sum().reset_index().rename(columns={'PitStop':'race_pits_this_lap'})
ru = ru.sort_values(['Race','Year','LapNumber'])
ru['race_pits_next_lap'] = ru.groupby(['Race','Year'])['race_pits_this_lap'].shift(-1)
full = full.merge(ru[['Race','Year','LapNumber','race_pits_next_lap']],
                  on=['Race','Year','LapNumber'], how='left')
print('after race ctx:', full.shape)

after race ctx: (627305, 97)


In [9]:
# 2e. Driver-race progress
full['stints_done_so_far'] = full['Stint'] - 1
full['race_total_laps']    = full.groupby(RACE)['LapNumber'].transform('max')
full['laps_left_in_race']  = full['race_total_laps'] - full['LapNumber']
full['frac_laps_left']     = full['laps_left_in_race'] / (full['race_total_laps'] + 1e-6)

# Cumulative PitStops this driver has done in this race so far (exclusive of current row)
full['driver_pits_so_far'] = g['PitStop'].cumsum() - full['PitStop']
print('after race progress:', full.shape)

after race progress: (627305, 102)


In [10]:
# 2f. Categoricals
CAT_COLS = ['Driver','Compound','Race','Compound_prev','Compound_next','Compound_prev2','Compound_next2']
for c in CAT_COLS:
    full[c] = full[c].astype('category')

drop_cols = [ID, TARGET, 'is_test']
feature_cols = [c for c in full.columns if c not in drop_cols]

train_fe = full[full['is_test']==0].copy()
test_fe  = full[full['is_test']==1].copy()
y    = train_fe[TARGET].astype(int).values
X    = train_fe[feature_cols]
X_te = test_fe[feature_cols]
print(f'features: {len(feature_cols)}  train: {X.shape}  test: {X_te.shape}  pos: {y.mean():.4f}')

features: 99  train: (439140, 99)  test: (188165, 99)  pos: 0.1990


## 3. CV folds (StratifiedKFold matches the LB row-split setup)

In [11]:
folds = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(X, y))
print('folds ready')

folds ready


## 4. LightGBM

In [ ]:
lgb_params = dict(
    objective='binary', metric='auc',
    learning_rate=0.03,
    num_leaves=511,
    min_child_samples=30,
    feature_fraction=0.85,
    bagging_fraction=0.85, bagging_freq=1,
    lambda_l2=2.0,
    verbose=-1, seed=SEED, n_jobs=-1,
)
oof_lgb  = np.zeros(len(X))
pred_lgb = np.zeros(len(X_te))
for f, (tr, va) in enumerate(folds):
    t0 = time.time()
    dtr = lgb.Dataset(X.iloc[tr], y[tr], categorical_feature=CAT_COLS)
    dva = lgb.Dataset(X.iloc[va], y[va], categorical_feature=CAT_COLS)
    m = lgb.train(lgb_params, dtr, num_boost_round=6000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(150), lgb.log_evaluation(0)])
    oof_lgb[va] = m.predict(X.iloc[va], num_iteration=m.best_iteration)
    pred_lgb   += m.predict(X_te, num_iteration=m.best_iteration) / N_FOLDS
    print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof_lgb[va]):.5f}  iter={m.best_iteration}  {time.time()-t0:.1f}s')
print(f'LGB OOF AUC: {roc_auc_score(y, oof_lgb):.5f}')

Training until validation scores don't improve for 150 rounds


## 5. CatBoost

In [ ]:
X_cb, X_te_cb = X.copy(), X_te.copy()
for c in CAT_COLS:
    X_cb[c]    = X_cb[c].astype(str).fillna('NA')
    X_te_cb[c] = X_te_cb[c].astype(str).fillna('NA')

oof_cb  = np.zeros(len(X))
pred_cb = np.zeros(len(X_te))
for f, (tr, va) in enumerate(folds):
    t0 = time.time()
    m = CatBoostClassifier(
        iterations=5000, learning_rate=0.05, depth=8,
        l2_leaf_reg=3.0, bagging_temperature=0.8, random_strength=1.0,
        eval_metric='AUC', cat_features=CAT_COLS,
        random_seed=SEED, early_stopping_rounds=150,
        verbose=0, task_type='CPU',
    )
    m.fit(X_cb.iloc[tr], y[tr], eval_set=(X_cb.iloc[va], y[va]), use_best_model=True)
    oof_cb[va] = m.predict_proba(X_cb.iloc[va])[:,1]
    pred_cb   += m.predict_proba(X_te_cb)[:,1] / N_FOLDS
    print(f'  fold {f+1}: AUC={roc_auc_score(y[va], oof_cb[va]):.5f}  iter={m.tree_count_}  {time.time()-t0:.1f}s')
print(f'CB OOF AUC: {roc_auc_score(y, oof_cb):.5f}')

## 6. Weighted blend (best single OR weighted by OOF AUC²)

In [ ]:
auc_lgb = roc_auc_score(y, oof_lgb)
auc_cb  = roc_auc_score(y, oof_cb)

# weighted on rank with weight = AUC² (rewards the stronger model more)
w_lgb, w_cb = auc_lgb**2, auc_cb**2
r_lgb_oof, r_cb_oof = rankdata(oof_lgb), rankdata(oof_cb)
r_lgb_te,  r_cb_te  = rankdata(pred_lgb), rankdata(pred_cb)
oof_blend  = (w_lgb * r_lgb_oof + w_cb * r_cb_oof) / (w_lgb + w_cb)
pred_blend = (w_lgb * r_lgb_te  + w_cb * r_cb_te ) / (w_lgb + w_cb)
auc_blend  = roc_auc_score(y, oof_blend)

print(f'LGB   AUC: {auc_lgb:.5f}')
print(f'CB    AUC: {auc_cb:.5f}')
print(f'BLEND AUC: {auc_blend:.5f}')

best_name = max([('lgb',auc_lgb),('cb',auc_cb),('blend',auc_blend)], key=lambda x:x[1])
print(f'\nBest: {best_name[0]} @ {best_name[1]:.5f}')
final_pred = {'lgb':pred_lgb,'cb':pred_cb,'blend':pred_blend}[best_name[0]]

## 7. Submission

In [ ]:
sub_out = test_fe[[ID]].copy()
sub_out[TARGET] = final_pred
sub_out.to_csv('submission.csv', index=False)
print('wrote submission.csv', sub_out.shape)
sub_out.head()